In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Optional, Iterable, Tuple
import joblib

In [2]:
TARGET_COLUMNS = [
    "Dry_Clover_g",
    "Dry_Dead_g",
    "Dry_Green_g",
    "Dry_Total_g",
    "GDM_g",
]

TARGET_WEIGHTS = {
    "Dry_Clover_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Green_g": 0.1,
    "Dry_Total_g": 0.5,
    "GDM_g": 0.2,
}

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        print("gpu")
        return torch.device("cuda")
    return torch.device("cpu")

def build_model(
    architecture: str = "resnet18",
    num_targets: int = 5,
    weights: Optional[object] = None,
) -> nn.Module:
    architecture = architecture.lower()
    if architecture == "resnet18":
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_targets)
    elif architecture == "densenet121":
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_targets)
    elif architecture == "efficientnet_b0":
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_targets)
    else:
        raise ValueError("Unknown architecture. Use 'resnet18' or 'densenet121'.")
    return model

class WeightedMSELoss(nn.Module):
    def __init__(self, weights: Optional[Iterable[float]] = None, device: Optional[torch.device] = None):
        super().__init__()
        if weights is None:
            weights = [TARGET_WEIGHTS[col] for col in TARGET_COLUMNS]
        self.device = device or get_device()
        self.weights = torch.tensor(weights, dtype=torch.float32).to(self.device)
        self.mse = nn.MSELoss(reduction="none")
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        loss = self.mse(pred, target)
        weighted_loss = loss * self.weights
        return weighted_loss.mean()

class WeightedHuberLoss(nn.Module):
    def __init__(self, device, target_weights, beta=1.0):
        super().__init__()
        self.register_buffer("weights", torch.tensor(target_weights, dtype=torch.float32, device=device))
        self.huber = nn.SmoothL1Loss(reduction="none", beta=beta)

    def forward(self, preds, targets):
        loss_per_elem = self.huber(preds, targets)          
        weighted = loss_per_elem * self.weights             
        return weighted.mean()

def create_kfold(n_splits: int = 5, shuffle: bool = True, random_state: int = 42) -> KFold:
    return KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

def scale_targets(
    dataframe,
    target_cols: Optional[Iterable[str]] = None,
    scaler: Optional[StandardScaler] = None,
) -> Tuple[object, StandardScaler]:
    if target_cols is None:
        target_cols = TARGET_COLUMNS
    scaler = scaler or StandardScaler()
    dataframe = dataframe.copy()
    dataframe[list(target_cols)] = scaler.fit_transform(dataframe[list(target_cols)])
    return dataframe, scaler

class ImageDataset(Dataset):
    def __init__(self, dataframe, transform=None, target_columns=None, has_targets=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.target_columns = list(target_columns or TARGET_COLUMNS)
        self.has_targets = has_targets

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index: int):
        img_path = self.dataframe.loc[index, "full_path"]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.has_targets:
            targets = self.dataframe.loc[index, self.target_columns].values.astype("float32")
            targets = torch.tensor(targets, dtype=torch.float32)
            return image, targets
        else:
            # For test set: no targets available
            return image

def train_regression(
    model,
    num_epochs,
    train_dl,
    valid_dl,
    loss_fn,
    optimizer,
    device,
    scheduler=None,
    use_amp=False,
    patience=30,              
    ckpt_path=None,           
):
    loss_hist_train = []
    loss_hist_valid = []

    use_cuda_amp = (use_amp and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda_amp)

    for epoch in range(num_epochs):
        model.train()
        running = 0.0

        for x_batch, y_batch in train_dl:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if use_cuda_amp:
                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    pred = model(x_batch)
                    loss = loss_fn(pred, y_batch)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = model(x_batch)
                loss = loss_fn(pred, y_batch)
                loss.backward()
                optimizer.step()

            if scheduler is not None:
                scheduler.step()

            running += loss.item()

        train_loss = running / max(1, len(train_dl))
        loss_hist_train.append(train_loss)

        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for x_batch, y_batch in valid_dl:
                x_batch = x_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)

                if use_cuda_amp:
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        pred = model(x_batch)
                        vloss = loss_fn(pred, y_batch)
                else:
                    pred = model(x_batch)
                    vloss = loss_fn(pred, y_batch)

                val_running += vloss.item()

        val_loss = val_running / max(1, len(valid_dl))
        loss_hist_valid.append(val_loss)

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch+1:2d}/{num_epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"LR: {lr_now:.2e}"
        )

    return loss_hist_train, loss_hist_valid


def evaluate_with_tta(model, loader, device, has_targets=True):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            if has_targets:
                x, y = batch
                all_targets.append(y.cpu())
            else:
                x = batch

            x = x.to(device)
            out_orig = model(x)
            out_hflip = model(torch.flip(x, dims=[3]))
            out_vflip = model(torch.flip(x, dims=[2]))
            avg_preds = (out_orig + out_hflip + out_vflip) / 3.0
            all_preds.append(avg_preds.cpu())

    preds = torch.cat(all_preds)
    if has_targets:
        targets = torch.cat(all_targets)
        return preds, targets
    return preds


def inverse_transform_preds_targets(preds, targets, scaler):
    preds_np = preds.cpu().numpy()
    targets_np = targets.cpu().numpy()
    preds_inv = scaler.inverse_transform(preds_np)
    targets_inv = scaler.inverse_transform(targets_np)
    return (torch.tensor(preds_inv, dtype=torch.float32), torch.tensor(targets_inv, dtype=torch.float32))

def calculate_global_weighted_r2(preds: torch.Tensor, targets: torch.Tensor) -> float:
    if preds.device != torch.device("cpu"):
        preds = preds.cpu()
    if targets.device != torch.device("cpu"):
        targets = targets.cpu()
    preds_np = preds.numpy()
    targets_np = targets.numpy()
    target_weights = TARGET_WEIGHTS
    n_samples = len(targets_np)
    flat_targets = targets_np.flatten()
    flat_preds = preds_np.flatten()
    weights = np.array([
        target_weights["Dry_Clover_g"],
        target_weights["Dry_Dead_g"],
        target_weights["Dry_Green_g"],
        target_weights["Dry_Total_g"],
        target_weights["GDM_g"],
    ])
    flat_weights = np.tile(weights, n_samples)
    y_bar_w = np.average(flat_targets, weights=flat_weights)
    ss_res = np.sum(flat_weights * (flat_targets - flat_preds) ** 2)
    ss_tot = np.sum(flat_weights * (flat_targets - y_bar_w) ** 2)
    if ss_tot == 0:
        return 1.0 if ss_res == 0 else 0.0
    return 1 - (ss_res / ss_tot)

DEVICE = get_device()
BASE_DIR = "/kaggle/input/csiro-biomass"
TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-4
SEED = 42
N_SPLITS = 5

seed_everything(SEED)

df_raw = pd.read_csv(TRAIN_CSV)
df_wide = df_raw.pivot(index='image_path', columns='target_name', values='target').reset_index()
df_wide["full_path"] = df_wide["image_path"].apply(lambda x: os.path.join(BASE_DIR, x))

df_train_part, df_holdout = train_test_split(df_wide, test_size=0.1, random_state=SEED)
df_train_part, scaler = scale_targets(df_train_part)
df_holdout, _ = scale_targets(df_holdout, scaler=scaler)

os.makedirs("/kaggle/working", exist_ok=True)
joblib.dump(scaler, "/kaggle/working/target_scaler_densenet.pkl")

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


gpu


In [3]:
kfold = create_kfold(n_splits=N_SPLITS, random_state=SEED)
criterion = WeightedHuberLoss(
    device=DEVICE,
    target_weights=[TARGET_WEIGHTS[c] for c in TARGET_COLUMNS]
)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(df_train_part)):
    print(f"--- Starting Fold {fold} (DenseNet121) ---")
    df_train_fold = df_train_part.iloc[train_idx]
    df_val_fold = df_train_part.iloc[val_idx]
    train_loader = DataLoader(ImageDataset(df_train_fold, transform=transform_train), batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(ImageDataset(df_val_fold, transform=transform_val), batch_size=BATCH_SIZE)
    model = build_model(architecture="densenet121", weights= None).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))
    train_regression(
        model=model,
        num_epochs=EPOCHS,
        train_dl=train_loader,
        valid_dl=valid_loader,
        loss_fn=criterion,
        optimizer=optimizer,
        device=DEVICE,
        scheduler=scheduler,
        use_amp=True
    )
    preds, targets = evaluate_with_tta(model, valid_loader, DEVICE)
    preds, targets = inverse_transform_preds_targets(preds, targets, scaler)
    fold_r2 = calculate_global_weighted_r2(preds, targets)
    print(f"Fold {fold} Weighted R2 (TTA): {fold_r2:.4f}")
    fold_metrics.append(fold_r2)
    torch.save(model.state_dict(), f"/kaggle/working/densenet121_fold_{fold}.pth")

print(f"\nAverage CV Weighted R2: {np.mean(fold_metrics):.4f}")

print("\n--- Running Ensemble Evaluation on Hold-out Set ---")
holdout_loader = DataLoader(ImageDataset(df_holdout, transform=transform_val), batch_size=BATCH_SIZE)
all_preds = []
for fold in range(N_SPLITS):
    m = build_model("densenet121", weights=None).to(DEVICE)
    m.load_state_dict(torch.load(f"/kaggle/working/densenet121_fold_{fold}.pth", map_location=DEVICE))
    preds, _ = evaluate_with_tta(m, holdout_loader, DEVICE)
    all_preds.append(preds.numpy())
    
ensemble_preds = np.mean(all_preds, axis=0)
ensemble_preds = scaler.inverse_transform(ensemble_preds)

_, ground_truth = evaluate_with_tta(m, holdout_loader, DEVICE)
ground_truth_inv = scaler.inverse_transform(ground_truth.numpy())
ensemble_r2 = calculate_global_weighted_r2(torch.tensor(ensemble_preds, dtype=torch.float32), torch.tensor(ground_truth_inv, dtype=torch.float32))
print(f"Ensemble Hold-out Weighted R2 (with TTA): {ensemble_r2:.4f}")

--- Starting Fold 0 (DenseNet121) ---
Epoch  1/100 | Train Loss: 0.073509 | Val Loss: 0.061652 | LR: 4.27e-06
Epoch  2/100 | Train Loss: 0.070480 | Val Loss: 0.059549 | LR: 5.06e-06
Epoch  3/100 | Train Loss: 0.068350 | Val Loss: 0.057088 | LR: 6.37e-06
Epoch  4/100 | Train Loss: 0.066853 | Val Loss: 0.054611 | LR: 8.18e-06
Epoch  5/100 | Train Loss: 0.064623 | Val Loss: 0.052299 | LR: 1.05e-05
Epoch  6/100 | Train Loss: 0.062261 | Val Loss: 0.050398 | LR: 1.32e-05
Epoch  7/100 | Train Loss: 0.061800 | Val Loss: 0.049415 | LR: 1.64e-05
Epoch  8/100 | Train Loss: 0.059017 | Val Loss: 0.048671 | LR: 2.00e-05
Epoch  9/100 | Train Loss: 0.057452 | Val Loss: 0.048360 | LR: 2.39e-05
Epoch 10/100 | Train Loss: 0.054509 | Val Loss: 0.048791 | LR: 2.82e-05
Epoch 11/100 | Train Loss: 0.055358 | Val Loss: 0.048617 | LR: 3.27e-05
Epoch 12/100 | Train Loss: 0.052163 | Val Loss: 0.047961 | LR: 3.74e-05
Epoch 13/100 | Train Loss: 0.051064 | Val Loss: 0.047413 | LR: 4.23e-05
Epoch 14/100 | Train Loss:

In [4]:
test_csv = os.path.join(BASE_DIR, "test.csv")
df_test = pd.read_csv(test_csv)

# FIX: remove duplicates
df_test = df_test.drop_duplicates(subset=["image_path"]).reset_index(drop=True)
df_test["full_path"] = df_test["image_path"].apply(
    lambda x: os.path.join(BASE_DIR, x)
)

test_loader = DataLoader(
    ImageDataset(df_test, transform=transform_val, has_targets=False),
    batch_size=BATCH_SIZE,
    shuffle=False
)

all_test_preds = []

for fold in range(N_SPLITS):
    m = build_model("densenet121", weights=None).to(DEVICE)
    m.load_state_dict(torch.load(f"/kaggle/working/densenet121_fold_{fold}.pth"))

    preds = evaluate_with_tta(m, test_loader, DEVICE, has_targets=False)
    all_test_preds.append(preds.numpy())

ensemble_test_preds = np.mean(all_test_preds, axis=0)
ensemble_test_preds = scaler.inverse_transform(ensemble_test_preds)

submission_rows = []

for i, image_path in enumerate(df_test["image_path"]):
    image_id = os.path.splitext(os.path.basename(image_path))[0]

    for j, target_name in enumerate(TARGET_COLUMNS):
        submission_rows.append({
            "sample_id": f"{image_id}__{target_name}",
            "target": float(ensemble_test_preds[i, j])
        })

submission_df = pd.DataFrame(submission_rows)

# Sanity checks
assert submission_df["sample_id"].duplicated().sum() == 0
assert len(submission_df) == len(df_test) * 5

save_path = "/kaggle/working/submission.csv"

submission_df.to_csv(save_path, index=False)

print("✅ Saved submission file at:", save_path)

# Show first 10 rows so you can visually confirm format
print("\n=== Preview of submission.csv ===")
print(submission_df.head(10))

# Confirm file appears in the directory
print("\n=== Files in /kaggle/working ===")
import os
print(os.listdir("/kaggle/working"))

print("Submission saved correctly!")
print(submission_df.head())

✅ Saved submission file at: /kaggle/working/submission.csv

=== Preview of submission.csv ===
                    sample_id     target
0  ID1001187975__Dry_Clover_g   5.114334
1    ID1001187975__Dry_Dead_g  32.986652
2   ID1001187975__Dry_Green_g  12.897117
3   ID1001187975__Dry_Total_g  48.623470
4         ID1001187975__GDM_g  18.610285

=== Files in /kaggle/working ===
['densenet121_fold_3.pth', 'densenet121_fold_0.pth', 'densenet121_fold_4.pth', 'submission.csv', 'densenet121_fold_2.pth', 'densenet121_fold_1.pth', '__notebook__.ipynb', 'target_scaler_densenet.pkl']
Submission saved correctly!
                    sample_id     target
0  ID1001187975__Dry_Clover_g   5.114334
1    ID1001187975__Dry_Dead_g  32.986652
2   ID1001187975__Dry_Green_g  12.897117
3   ID1001187975__Dry_Total_g  48.623470
4         ID1001187975__GDM_g  18.610285


In [5]:
import pandas as pd
import numpy as np

sample = pd.read_csv("/kaggle/input/csiro-biomass/sample_submission.csv")
sub = pd.read_csv("/kaggle/working/submission.csv")

print("SAMPLE columns:", sample.columns.tolist())
print("SUB columns   :", sub.columns.tolist())
print("SAMPLE shape:", sample.shape, " SUB shape:", sub.shape)

assert list(sub.columns) == list(sample.columns), "❌ Column names/order mismatch vs sample_submission"

missing = set(sample["sample_id"]) - set(sub["sample_id"])
extra   = set(sub["sample_id"]) - set(sample["sample_id"])
print("Missing sample_id:", len(missing))
print("Extra sample_id  :", len(extra))
assert len(missing) == 0, f"❌ Missing {len(missing)} sample_ids"
assert len(extra) == 0, f"❌ Extra {len(extra)} sample_ids"

assert np.isfinite(sub["target"]).all(), "❌ NaN/Inf in target"
print("✅ Submission matches sample_submission schema & IDs.")
display(sub.head())

SAMPLE columns: ['sample_id', 'target']
SUB columns   : ['sample_id', 'target']
SAMPLE shape: (5, 2)  SUB shape: (5, 2)
Missing sample_id: 0
Extra sample_id  : 0
✅ Submission matches sample_submission schema & IDs.


,sample_id,target
0,ID1001187975__Dry_Clover_g,5.114334
1,ID1001187975__Dry_Dead_g,32.986652
2,ID1001187975__Dry_Green_g,12.897117
3,ID1001187975__Dry_Total_g,48.623470
4,ID1001187975__GDM_g,18.610285
